In [5]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "MMSR":
    if (PROJECT_ROOT / "MMSR").is_dir():
        PROJECT_ROOT = PROJECT_ROOT / "MMSR"
    elif (PROJECT_ROOT.parent / "src").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT / "src").is_dir():
        PROJECT_ROOT = PROJECT_ROOT
    else:
        raise FileNotFoundError(
            "Could not locate the MMSR project root. Current directory: "
            + str(PROJECT_ROOT)
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.query_config import QUERIES, TOP_K

from src.tfidf_retrieval import (
    DATA_PATH,
    RESULTS_PATH,
    TEXT_COLUMN,
    EXPECTED_ROW_COUNT,
    TFIDF_CONFIG,
    load_and_validate_corpus,
    build_vectorizer,
    run_tfidf_retrieval,
)

print("Project root:", PROJECT_ROOT)
print("Imports successful.")

Project root: c:\Users\Subhash\Desktop\SEMESTER-7\SpeechNLP\MMSR
Imports successful.


In [6]:
df = load_and_validate_corpus()

print("Corpus loaded successfully.")
print("Number of records:", len(df))
print("Text column:", TEXT_COLUMN)

Corpus loaded successfully.
Number of records: 94
Text column: clean_synopsis


In [7]:
vectorizer = build_vectorizer()

tfidf_matrix = vectorizer.fit_transform(
    df[TEXT_COLUMN].astype(str)
)

print("TF-IDF model fitted successfully.")
print("Matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF model fitted successfully.
Matrix shape: (94, 20422)
Vocabulary size: 20422


In [8]:
all_results = []

for query_id, query in enumerate(QUERIES, start=1):
    query_vector = vectorizer.transform([query])
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).ravel()

    ranking = sorted(
        range(len(df)),
        key=lambda i: (-similarity_scores[i], df.iloc[i]["source_row_id"])
    )[:TOP_K]

    for rank, index in enumerate(ranking, start=1):
        movie = df.iloc[index]

        all_results.append({
            "query_id": query_id,
            "query": query,
            "rank": rank,
            "source_row_id": movie["source_row_id"],
            "title": movie["title"],
            "genre": movie["genre"],
            "rating": movie["rating"],
            "similarity_score": float(similarity_scores[index])
        })

tfidf_results_df = pd.DataFrame(
    all_results,
    columns=[
        "query_id",
        "query",
        "rank",
        "source_row_id",
        "title",
        "genre",
        "rating",
        "similarity_score"
    ]
)

print("Retrieval completed.")
print("Total result rows:", len(tfidf_results_df))

Retrieval completed.
Total result rows: 30


In [9]:
print(tfidf_results_df.groupby("query_id").size())

query_id
1    5
2    5
3    5
4    5
5    5
6    5
dtype: int64


In [10]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

tfidf_results_df.to_csv(
    RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Results saved successfully.")
print("File:", RESULTS_PATH)
print("Rows:", len(tfidf_results_df))

Results saved successfully.
File: C:\Users\Subhash\Desktop\SEMESTER-7\SpeechNLP\MMSR\results\tfidf\tfidf_results.csv
Rows: 30


In [11]:
saved_results = pd.read_csv(RESULTS_PATH)

print("File exists:", RESULTS_PATH.is_file())
print("Saved rows:", len(saved_results))
print("Columns:", list(saved_results.columns))

File exists: True
Saved rows: 30
Columns: ['query_id', 'query', 'rank', 'source_row_id', 'title', 'genre', 'rating', 'similarity_score']
